# Training inspection

Two modes:

1. **Browse saved val grids** — fast: just walks `runs/<run>/val/val_step_*.png` and shows them inline. Use this for routine progress monitoring.
2. **Live one-shot rollout** — slow: loads the latest checkpoint and runs the diffusion sampler on the val batch. Use sparingly; the saved PNGs are normally enough.

Set `RUN_DIR` below to the run you want.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

REPO = Path('..').resolve()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

RUN_DIR = REPO / 'runs' / 'dev01'
DATA_DIR = Path('/opt/dlami/nvme/cs2-data')

print('RUN_DIR:', RUN_DIR)
print('DATA_DIR:', DATA_DIR)
print('latest.pt exists:', (RUN_DIR / 'latest.pt').exists())
print('val grids:', len(sorted((RUN_DIR / 'val').glob('val_step_*.png'))) if (RUN_DIR / 'val').exists() else 0)

## 1. Browse saved val grids

In [ ]:
from IPython.display import Image, display
import matplotlib.pyplot as plt
from PIL import Image as PILImage

val_dir = RUN_DIR / 'val'
pngs = sorted(val_dir.glob('val_step_*.png'))
print(f'Found {len(pngs)} val grids')

# Show the most recent N
N = 5
for p in pngs[-N:]:
    print(p.name)
    display(Image(filename=str(p)))

In [ ]:
# Plot the val_mse / val_psnr trend by parsing PNG filenames + reading the run config
# (we don't reload the model — we just want the curve quickly).
import json, re
import matplotlib.pyplot as plt

cfg_path = RUN_DIR / 'config.json'
if cfg_path.exists():
    cfg = json.loads(cfg_path.read_text())
    print('preset:', cfg.get('preset'), 'resolved:', cfg.get('preset_resolved'))

# Note: training steps are stored in the PNG filenames; the actual val_mse/psnr is in W&B.
# This cell is a placeholder for offline inspection without W&B access.
steps = [int(re.search(r'val_step_(\d+)', p.name).group(1)) for p in pngs]
print('val checkpoints at steps:', steps)

## 2. Live one-shot rollout (slow)

Loads `latest.pt` and runs the diffusion sampler on the same fixed val batch the trainer uses, then displays the grid inline (without saving to disk).

In [ ]:
import torch
from src.action_encoder import NUM_ACTIONS
from src.dataset import CSDataset, collate_diamond
from src.diamond import Denoiser, DenoiserConfig, InnerModelConfig, SigmaDistributionConfig
from src.visualize import run_validation, rollout_one_step, make_grid

ckpt_path = RUN_DIR / 'latest.pt'
ckpt = torch.load(ckpt_path, map_location='cuda', weights_only=False)
args = ckpt['args']
preset = args.get('preset_resolved') or {
    'cond_channels': 256, 'base_channels': 64, 'depths': [2,2,2,2], 'attn_depths': [0,0,1,1], 'resize': (36, 64),
}
print('checkpoint step:', ckpt['step'], 'best_val_mse:', ckpt.get('best_val_mse'))

# Re-build the model with the saved config
inner = InnerModelConfig(
    img_channels=3,
    num_steps_conditioning=args['cond_frames'],
    cond_channels=preset['cond_channels'],
    depths=list(preset['depths']),
    channels=[preset['base_channels'] * m for m in (1, 2, 4, 8)],
    attn_depths=list(preset['attn_depths']),
    num_actions=NUM_ACTIONS,
    is_upsampler=False,
)
cfg = DenoiserConfig(inner_model=inner, sigma_data=0.5, sigma_offset_noise=0.1, noise_previous_obs=True)
model = Denoiser(cfg).to('cuda')
model.setup_training(SigmaDistributionConfig(loc=-1.2, scale=1.2, sigma_min=2e-3, sigma_max=20))
model.load_state_dict(ckpt['model'])
model.eval()
print(f'loaded model with {sum(p.numel() for p in model.parameters())/1e6:.1f}M params')

In [ ]:
T = args['cond_frames'] + 1 + args['num_autoregressive_steps']
val_ds = CSDataset(
    data_path=DATA_DIR, split='val', T=T, stride=args['stride'],
    resize=tuple(preset['resize']), mode='diamond',
)
print(f'val: {len(val_ds)} windows / {len(val_ds.samples)} clips')

# Same fixed indices the trainer uses
rng = torch.Generator().manual_seed(123)
val_indices = torch.randperm(len(val_ds), generator=rng)[: args['val_batch_size']].tolist()
print('val indices:', val_indices)

batch = collate_diamond([val_ds[i] for i in val_indices]).to('cuda')
pred = rollout_one_step(model, batch, num_denoising_steps=args.get('val_denoise_steps', 10))
fig = make_grid(batch, pred, n_cond=args['cond_frames'], max_rows=args['val_rows'],
                title=f"step {ckpt['step']} (live rollout)")
fig.set_dpi(110)
fig